# Task 2 — Quote Scraper & Analyzer (Web Scraping + OOP + Collections)

**Solution notebook.** This notebook first tries to scrape the live practice site
`https://quotes.toscrape.com`. If your network can't reach it (blocked lab wifi,
offline classroom, etc.) it automatically falls back to `quotes_fallback.csv` —
exactly like the assignment brief describes. Every cell is commented.

In [1]:
# Imports
import time
import requests
from bs4 import BeautifulSoup
from collections import Counter, defaultdict
import pandas as pd

## Part A — Ethics & Setup

In [2]:
BASE_URL = "https://quotes.toscrape.com"

# Always check robots.txt before scraping ANY site — this one is built for
# practice and explicitly allows scraping, but the habit matters more than
# the specific site.
robots_url = f"{BASE_URL}/robots.txt"
try:
    robots = requests.get(robots_url, timeout=5)
    print(f"robots.txt status: {robots.status_code}")
    print(robots.text[:300])
except Exception as e:
    print(f"Could not reach robots.txt ({e}) — will use offline fallback below.")

robots.txt status: 404
<!doctype html>
<html lang=en>
<title>404 Not Found</title>
<h1>Not Found</h1>
<p>The requested URL was not found on the server. If you entered the URL manually please check your spelling and try again.</p>



In [3]:
# Confirm the homepage responds before we try to parse anything.
# We store the result so later cells know whether to scrape live or fall back.
SCRAPE_LIVE = False
try:
    resp = requests.get(BASE_URL, timeout=5)
    if resp.status_code == 200:
        SCRAPE_LIVE = True
        print("Live site reachable — will scrape.")
    else:
        print(f"Site responded with {resp.status_code} — using offline fallback CSV instead.")
except Exception as e:
    print(f"Network error ({e}) — using offline fallback CSV instead.")

Live site reachable — will scrape.


## Part B — Model the Data with OOP

In [4]:
class Quote:
    """Represents a single quote scraped from the site.

    __eq__/__hash__ are defined on (text, author) so that two Quote objects
    describing the same quote are treated as equal — this is what lets us
    dedupe a list of Quote objects using a plain Python set() later.
    """

    def __init__(self, text, author, tags):
        self.text = text
        self.author = author
        self.tags = tags  # list[str]

    def __repr__(self):
        # A short, readable representation for debugging/printing
        return f"Quote(author={self.author!r}, tags={len(self.tags)})"

    def __eq__(self, other):
        if not isinstance(other, Quote):
            return NotImplemented
        return (self.text, self.author) == (other.text, other.author)

    def __hash__(self):
        # Must match __eq__: same (text, author) -> same hash
        return hash((self.text, self.author))


# Quick smoke test
q1 = Quote("Be yourself.", "Anon", ["identity"])
q2 = Quote("Be yourself.", "Anon", ["identity"])
print(q1)
print("q1 == q2:", q1 == q2)          # True -> __eq__ works
print("hashable:", {q1, q2})           # a set collapses them into ONE item

Quote(author='Anon', tags=1)
q1 == q2: True
hashable: {Quote(author='Anon', tags=1)}


## Part C — Scrape With Pagination (or load fallback data)

In [5]:
def parse_page(html):
    """Extract every Quote object from one page's HTML."""
    soup = BeautifulSoup(html, "html.parser")
    quotes_on_page = []

    for block in soup.select("div.quote"):
        text = block.select_one("span.text").get_text(strip=True).strip('"\u201c\u201d')
        author = block.select_one("small.author").get_text(strip=True)
        tags = [tag.get_text(strip=True) for tag in block.select("a.tag")]
        quotes_on_page.append(Quote(text, author, tags))

    # Find the "Next ->" link, if any, so the caller knows whether to keep paginating
    next_link = soup.select_one("li.next > a")
    next_href = next_link["href"] if next_link else None

    return quotes_on_page, next_href

In [6]:
all_quotes = []

if SCRAPE_LIVE:
    # --- LIVE SCRAPE PATH ---
    url = BASE_URL + "/"
    page_num = 1
    while url:
        print(f"Scraping page {page_num}: {url}")
        r = requests.get(url, timeout=10)
        page_quotes, next_href = parse_page(r.text)
        all_quotes.extend(page_quotes)

        url = BASE_URL + next_href if next_href else None
        page_num += 1
        time.sleep(0.5)  # be a polite scraper, even on a practice site

    print(f"\nScraped {len(all_quotes)} quotes across {page_num - 1} pages.")

else:
    # --- OFFLINE FALLBACK PATH ---
    # Load the original practice dataset and build Quote objects the same way,
    # so every step below behaves identically regardless of which path ran.
    fallback_df = pd.read_csv("quotes_fallback.csv")
    for row in fallback_df.itertuples(index=False):
        tags = row.tags.split("|")
        all_quotes.append(Quote(row.text, row.author, tags))

    print(f"Loaded {len(all_quotes)} quotes from quotes_fallback.csv (offline mode).")

all_quotes[:3]

Scraping page 1: https://quotes.toscrape.com/
Scraping page 2: https://quotes.toscrape.com/page/2/
Scraping page 3: https://quotes.toscrape.com/page/3/
Scraping page 4: https://quotes.toscrape.com/page/4/
Scraping page 5: https://quotes.toscrape.com/page/5/
Scraping page 6: https://quotes.toscrape.com/page/6/
Scraping page 7: https://quotes.toscrape.com/page/7/
Scraping page 8: https://quotes.toscrape.com/page/8/
Scraping page 9: https://quotes.toscrape.com/page/9/
Scraping page 10: https://quotes.toscrape.com/page/10/

Scraped 100 quotes across 10 pages.


[Quote(author='Albert Einstein', tags=4),
 Quote(author='J.K. Rowling', tags=2),
 Quote(author='Albert Einstein', tags=5)]

## Part D — Analyze With Collections

In [7]:
# Deduplicate using a set — relies entirely on the __eq__/__hash__ we wrote earlier.
unique_quotes = list(set(all_quotes))
print(f"Before dedup: {len(all_quotes)}   After dedup: {len(unique_quotes)}")

Before dedup: 100   After dedup: 100


In [8]:
# Flatten every quote's tag list into one big list, then count with Counter.
all_tags = [tag for quote in unique_quotes for tag in quote.tags]
tag_counts = Counter(all_tags)

print("Top 5 most common tags:")
for tag, count in tag_counts.most_common(5):
    print(f"  {tag}: {count}")

Top 5 most common tags:
  love: 14
  life: 13
  inspirational: 13
  humor: 12
  books: 11


In [9]:
# Group quotes by author using defaultdict(list) — no need to check
# "does this key exist yet" before appending.
quotes_by_author = defaultdict(list)
for quote in unique_quotes:
    quotes_by_author[quote.author].append(quote)

# Which author has the most quotes on this site/dataset?
author_counts = {author: len(qs) for author, qs in quotes_by_author.items()}
top_author = max(author_counts, key=author_counts.get)

print(f"Author with the most quotes: {top_author} ({author_counts[top_author]} quotes)")
print("\nFull breakdown:")
for author, count in sorted(author_counts.items(), key=lambda x: -x[1]):
    print(f"  {author}: {count}")

Author with the most quotes: Albert Einstein (10 quotes)

Full breakdown:
  Albert Einstein: 10
  J.K. Rowling: 9
  Marilyn Monroe: 7
  Mark Twain: 6
  Dr. Seuss: 6
  Jane Austen: 5
  C.S. Lewis: 5
  Bob Marley: 3
  Eleanor Roosevelt: 2
  Mother Teresa: 2
  Ernest Hemingway: 2
  George R.R. Martin: 2
  Ralph Waldo Emerson: 2
  Suzanne Collins: 2
  Charles Bukowski: 2
  André Gide: 1
  Jim Henson: 1
  J.D. Salinger: 1
  Friedrich Nietzsche: 1
  Garrison Keillor: 1
  Thomas A. Edison: 1
  Pablo Neruda: 1
  Elie Wiesel: 1
  Jimi Hendrix: 1
  Harper Lee: 1
  Allen Saunders: 1
  George Carlin: 1
  J.M. Barrie: 1
  Haruki Murakami: 1
  Alfred Tennyson: 1
  E.E. Cummings: 1
  Steve Martin: 1
  Madeleine L'Engle: 1
  Martin Luther King Jr.: 1
  Stephenie Meyer: 1
  Charles M. Schulz: 1
  John Lennon: 1
  Jorge Luis Borges: 1
  Ayn Rand: 1
  Helen Keller: 1
  Douglas Adams: 1
  George Bernard Shaw: 1
  Alexandre Dumas fils: 1
  William Nicholson: 1
  Khaled Hosseini: 1
  James Baldwin: 1
  J.R.

## Part E — Export

In [10]:
# Build a list of plain dicts first — much easier to turn into a DataFrame
# than assembling columns one at a time.
records = [
    {
        "text": q.text,
        "author": q.author,
        "tags": "|".join(q.tags),  # join list into one CSV-friendly string
    }
    for q in unique_quotes
]

quotes_df = pd.DataFrame(records)
quotes_df.to_csv("scraped_quotes.csv", index=False)

print(f"Saved scraped_quotes.csv with {len(quotes_df)} rows.")
quotes_df.head()

Saved scraped_quotes.csv with 100 rows.


,text,author,tags
0,It is better to be hated for what you are than...,André Gide,life|love
1,′Classic′ - a book which people praise and don...,Mark Twain,books|classic|reading
2,"The person, be it gentleman or lady, who has n...",Jane Austen,aliteracy|books|classic|humor
3,"The truth."" Dumbledore sighed. ""It is a beauti...",J.K. Rowling,truth
4,A woman is like a tea bag; you never know how ...,Eleanor Roosevelt,misattributed-eleanor-roosevelt


## Bonus Challenge

- A `QuoteCollection` class wrapping the list with `.filter_by_author()` and `.top_tags()` methods
- Re-running the scrape/load and confirming the dedup `set` catches repeats

In [11]:
class QuoteCollection:
    """Wraps a list of Quote objects behind a clean, reusable interface."""

    def __init__(self, quotes):
        self.quotes = list(quotes)

    def filter_by_author(self, author):
        return [q for q in self.quotes if q.author == author]

    def top_tags(self, n=5):
        tags = [tag for q in self.quotes for tag in q.tags]
        return Counter(tags).most_common(n)

    def __len__(self):
        return len(self.quotes)


collection = QuoteCollection(unique_quotes)
print(f"Collection size: {len(collection)}")
print(f"Top 3 tags: {collection.top_tags(3)}")
print(f"Quotes by '{top_author}': {len(collection.filter_by_author(top_author))}")

Collection size: 100
Top 3 tags: [('love', 14), ('life', 13), ('inspirational', 13)]
Quotes by 'Albert Einstein': 10


In [12]:
# Confirm dedup logic: merging the same data with itself should NOT double the count,
# because __eq__/__hash__ collapse the repeats.
doubled = all_quotes + all_quotes
deduped_again = set(doubled)
print(f"Doubled list length: {len(doubled)}")
print(f"After dedup: {len(deduped_again)}  (should match the unique_quotes count: {len(unique_quotes)})")

Doubled list length: 200
After dedup: 100  (should match the unique_quotes count: 100)
